In [16]:
pip install PyPDF2 paddleocr paddle

  Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 2.9 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 11.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 6.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.4/586.4 kB 27.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 MB 7.7 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.8/22.8 MB 9.2 MB/s  0:00:02m0:00:0100:01
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl (20.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [paddle] 7/10 [kintera]]
Note: you may need to restart the kernel to use updated packages.


# 01 · 文档解析（PDF → Markdown）

示例：`data/道通24年年报.pdf`

本节做三件事：
- **拆分 PDF**：大文件先按页数切片，提升解析稳定性
- **两路解析对比**：
  - MinerU API（返回结构化 Markdown）
  - PaddleOCR-VL-1.5（本地 doc-parser 输出 Markdown/JSON）
- **重点对比表格**：用“表格块数量 + 结构相似度”

运行前：
- `pip install -r RAG_project/requirements-rag312.txt`
- 设置 `MINERU_API_KEY`（MinerU）
- PaddleOCR-VL 初次运行可能较慢（会下载模型/初始化）


In [11]:
from __future__ import annotations

import json
import os
import re
import sys
from difflib import SequenceMatcher
from pathlib import Path

from dotenv import load_dotenv

#请修改根目录
PROJECT_ROOT = Path.home() / "Documents" / "AI-training"

load_dotenv(PROJECT_ROOT / ".env")
assert os.getenv("MINERU_API_KEY"), "MINERU_API_KEY 未加载，检查 .env 路径"

PDF_PATH = PROJECT_ROOT / "RAG_project" / "data" / "道通24年年报.pdf"
OUT_DIR = PROJECT_ROOT / "RAG_project" / "data" / "parsed" / "道通24年年报"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert PDF_PATH.exists(), f"找不到 PDF：{PDF_PATH}"
print("PDF:", PDF_PATH, "bytes=", PDF_PATH.stat().st_size)
print("OUT:", OUT_DIR)


PDF: /Users/mengbai/Documents/AI-training/RAG_project/data/道通24年年报.pdf bytes= 3455119
OUT: /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报


文件过大先拆分


In [12]:
# 1) 拆分 PDF（默认每 50 页一份）

from PyPDF2 import PdfReader, PdfWriter


def split_pdf(pdf_path: Path, out_dir: Path, pages_per_chunk: int = 50) -> list[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    reader = PdfReader(str(pdf_path))
    total = len(reader.pages)
    assert total > 0, "PDF 页数为 0"

    outs: list[Path] = []
    for start in range(0, total, pages_per_chunk):
        end = min(start + pages_per_chunk, total)
        w = PdfWriter()
        for p in range(start, end):
            w.add_page(reader.pages[p])
        out = out_dir / f"{pdf_path.stem}_p{start+1:04d}-{end:04d}.pdf"
        with open(out, "wb") as f:
            w.write(f)
        outs.append(out)
    return outs


SPLIT_DIR = OUT_DIR / "splits"
split_paths = split_pdf(PDF_PATH, SPLIT_DIR, pages_per_chunk=int(os.getenv("PAGES_PER_CHUNK", "50")))
print("splits:", len(split_paths))
print("first:", split_paths[0].name)


splits: 7
first: 道通24年年报_p0001-0050.pdf


# PaddleOCR-VL-1.5 解析

In [26]:
# 3) PaddleOCR-VL-1.5 解析（逐个分片跑，不启用 ocr-correct）
# 用当前 notebook 的 Python 环境，不依赖本地服务/固定路径

import sys
import subprocess

PADDLE_BASE_DIR = OUT_DIR / "paddleocr_vl"
PADDLE_BASE_DIR.mkdir(parents=True, exist_ok=True)

parser_script = PROJECT_ROOT / "RAG_project" / "parse_with_paddleocr_vl.py"
python_bin = sys.executable

assert parser_script.exists(), f"找不到脚本：{parser_script}"

for pdf_chunk in split_paths:
    chunk_out_dir = PADDLE_BASE_DIR / pdf_chunk.stem

    if any(chunk_out_dir.glob("*.md")):
        print("skip (exists):", chunk_out_dir)
        continue

    cmd = [
        python_bin,
        str(parser_script),
        str(pdf_chunk),
        "-o",
        str(PADDLE_BASE_DIR),
        "--merge-tables",
        "--relevel-titles",
    ]

    print("\n>>> Running:", pdf_chunk.name)

    res = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT),
        env={**os.environ, "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True"},
        text=True,
        capture_output=True,
    )

    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f"PaddleOCR-VL failed on {pdf_chunk.name}")

print("done:", PADDLE_BASE_DIR)

skip (exists): /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报/paddleocr_vl/道通24年年报_p0001-0050
skip (exists): /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报/paddleocr_vl/道通24年年报_p0051-0100
skip (exists): /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报/paddleocr_vl/道通24年年报_p0101-0150
skip (exists): /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报/paddleocr_vl/道通24年年报_p0151-0200
skip (exists): /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报/paddleocr_vl/道通24年年报_p0201-0250
skip (exists): /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报/paddleocr_vl/道通24年年报_p0251-0300
skip (exists): /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报/paddleocr_vl/道通24年年报_p0301-0306
done: /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报/paddleocr_vl


# 可选内容：MinerU VLM 解析

In [25]:
# 2) MinerU VLM 解析（mineru.net 官网 API，model_version="vlm"）
# 这一格单独运行也能拿到 .env 里的 MINERU_API_KEY

from dotenv import load_dotenv
import importlib

load_dotenv(PROJECT_ROOT / ".env", override=True)
mineru_key = os.getenv("MINERU_API_KEY")
assert mineru_key, f"MINERU_API_KEY 未加载：{PROJECT_ROOT / '.env'}"

sys.path.append(str(PROJECT_ROOT / "RAG_project"))

import mineru_process  # type: ignore
importlib.reload(mineru_process)
from mineru_process import parse_pdf_by_url, extract_markdown  # type: ignore

PDF_URL = "https://www.auteltech.cn/u/cms/wwwcn/202504/070921187kb3.pdf"

MINERU_DIR = OUT_DIR / "mineru"
MINERU_DIR.mkdir(parents=True, exist_ok=True)
mineru_md_path = MINERU_DIR / f"{PDF_PATH.stem}_vlm.md"

if mineru_md_path.exists():
    print("skip (exists):", mineru_md_path.name)
else:
    res = parse_pdf_by_url(PDF_URL, model_version="vlm")
    assert res, "MinerU 返回为空：检查 MINERU_API_KEY / 任务接口状态"
    md = extract_markdown(res)
    mineru_md_path.write_text(md, encoding="utf-8")
    print("saved:", mineru_md_path, "chars:", len(md))

  POST https://mineru.net/api/v4/extract/task url=https://www.auteltech.cn/u/cms/wwwcn/202504/070921187kb3.pdf...
  task_id: fed44f0a-b117-41eb-8d13-6a49cf1141b3, polling for result...
  State: pending, waiting...
  Progress: 7/306 pages
  Progress: 19/306 pages
  Progress: 29/306 pages
  Progress: 39/306 pages
  Progress: 49/306 pages
  Progress: 59/306 pages
  Progress: 69/306 pages
  Progress: 80/306 pages
  Progress: 90/306 pages
  Progress: 100/306 pages
saved: /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报/mineru/道通24年年报_vlm.md chars: 338843


# 课后练习

用 VLM + OCR的形式解析pdf

## 产物位置（你后续 chunk/入库会用到）

- 拆分后的 PDF：`data/parsed/道通24年年报/splits/`
- PaddleOCR-VL 输出：`data/parsed/道通24年年报/paddleocr_vl/<split_stem>/*.md`

下一节：打开 `01_data_02_chunk_ingest.ipynb`，把这些 Markdown 做 chunk 并写入 Chroma。


## 下一步

- 如果你想做“更严格”的表格准确率对比：
  - 先手工挑 2-3 页“有代表性的表格”（复杂表头、合并单元格、跨页表）
  - 把这些页作为固定回归集
  - 用单元格/行级规则（或 LLM）对齐后再算 precision/recall

在下一节 `01_data_02_chunk_ingest.ipynb` 我们会把解析后的 Markdown 直接做 chunk 并入库到 Chroma。
